# Dog Skin Disease Classification Model 

This models trains and evaluates a **MobileNetV2 transfer-learning model** for dog skin disease classification. It is structured for Google Colab and uses an uploaded ZIP dataset containing separate **train**, **valid**, and **test** folders. The comments explain each implementation step, including dataset loading, preprocessing, augmentation, model building, training, fine-tuning, testing, and exporting the trained model.


## 1. Upload and Extract Dataset

This section uploads the dataset ZIP file into Google Colab, extracts it into a working directory, and prints the folder structure. The output is useful for checking whether the dataset has the expected train/validation/test split and class folders before training begins.


In [ ]:
from google.colab import files
import zipfile, os

# STEP 1: UPLOAD AND EXTRACT DATASET

# PURPOSE: Load the training dataset from Google Drive into Colab
#
# DATASET STRUCTURE EXPECTED:
# archive.zip
#  ├── train/
#  │   ├── Healthy Skin/       (class folder with images)
#  │   ├── Dermatitis/
#  │   ├── Fungal Infection/
#  │   └── ... (other disease classes)
#  ├── valid/                   (validation split)
#  │   ├── Healthy Skin/
#  │   └── ...
#  └── test/                    (test split - final evaluation)
#      ├── Healthy Skin/
#      └── ...
#

# Step 1: Upload the dataset ZIP file
# In Google Colab, this opens a browser file picker where you select the archive.
# The ZIP file should contain the train/valid/test folder structure described above.
uploaded = files.upload()  # select your dataset archive, e.g., archive.zip

# Get the filename of the uploaded ZIP file
# files.upload() returns a dictionary where keys are uploaded filenames
zip_name = list(uploaded.keys())[0]
print(f" Uploaded file: {zip_name}")

# Extract the entire ZIP file into /content/dataset
# This path is used throughout the notebook as the main dataset location
extract_path = '/content/dataset'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)
    print(f"✓ Extracted to: {extract_path}")


# VERIFICATION: Check dataset structure and image counts
# This helps identify issues BEFORE training (e.g., missing classes, imbalanced data)


print("\n📊 DATASET STRUCTURE:")
print("=" * 70)

for split in ['train', 'valid', 'test']:
    path = f'{extract_path}/{split}'
    
    # Check if this split exists
    if os.path.exists(path):
        classes = sorted(os.listdir(path))
        print(f"\n📁 {split.upper()}/")
        total_images = 0
        
        # Count images in each class folder
        for class_name in classes:
            class_path = os.path.join(path, class_name)
            if os.path.isdir(class_path):
                image_count = len(os.listdir(class_path))
                total_images += image_count
                print(f"   └─ {class_name:<30s} : {image_count:4d} images")
        
        print(f"   {'─' * 45}")
        print(f"   {'Total':<30s} : {total_images:4d} images")
    else:
        print(f"\n  {split.upper()}/ folder not found!")

print("\n" + "=" * 70)

## 2. Import Libraries and Check Runtime

This section imports the main libraries required for model development. TensorFlow and Keras are used for deep learning, MobileNetV2 is used as the pretrained base model, NumPy supports numerical operations, and Matplotlib is used for visualising training performance.


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt


# STEP 2: IMPORT LIBRARIES AND CHECK RUNTIME ENVIRONMENT

#
# TensorFlow: Deep learning framework
# Keras: High-level neural network API (integrated into TensorFlow)
# MobileNetV2: Efficient pretrained model (1000 classes from ImageNet)
# NumPy: Numerical computations
# Matplotlib: Visualization
#


print("🔧 ENVIRONMENT SETUP")
print("=" * 70)

# Print TensorFlow version for reproducibility
# This helps others replicate the exact same results
print(f"TensorFlow version: {tf.__version__}")

# Check GPU availability
# Training on GPU is 10-100x faster than CPU for deep learning
# If empty list is returned in Colab: Runtime > Change runtime type > GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"✓ GPU Available: {gpu_devices[0].name}")
else:
    print(" No GPU detected. Training will be SLOW on CPU.")
    print("   → In Colab: Runtime > Change runtime type > GPU")

# Check CPU info for reference
cpu_devices = tf.config.list_physical_devices('CPU')
print(f"CPU: {cpu_devices[0].name if cpu_devices else 'Unknown'}")

print("=" * 70 + "\n")

## 3. Dataset Preprocessing and Data Augmentation

Images are resized to **224 × 224 pixels**, which is the expected input size for MobileNetV2. The notebook applies the official MobileNetV2 preprocessing function and uses data augmentation on the training set to improve generalisation.


In [ ]:

# STEP 3: DATASET PREPROCESSING & DATA AUGMENTATION

#
# WHY?: 
# • Images vary in size/quality → must standardize to 224×224
# • MobileNetV2 was trained on ImageNet with specific preprocessing
# • Data augmentation artificially expands dataset → reduces overfitting
# • Augmented variations teach model to be robust to real-world variations
#

# Main dataset location (created in previous step)
DATASET_PATH = '/content/dataset'

# Image size required by MobileNetV2
# MobileNetV2 was pretrained on ImageNet using 224×224 pixel images.
# Using different sizes requires retraining the normalization statistics.
IMG_SIZE = (224, 224)
print(f" Image size: {IMG_SIZE[0]}×{IMG_SIZE[1]} pixels")

# Batch size: number of images processed before model weights are updated
# Trade-off: 
#  - Larger batches (64, 128): Faster training, more memory
#  - Smaller batches (16, 32): Slower training, less memory, sometimes better convergence
# 32 is a good balance for most GPUs and datasets
BATCH_SIZE = 32
print(f" Batch size: {BATCH_SIZE} images per batch\n")

# ─────────────────────────────────────────────────────────────────────────────
# DATA AUGMENTATION CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Purpose: Create slightly modified versions of training images to:
#  1. Prevent overfitting (model doesn't memorize exact images)
#  2. Improve robustness (model handles real-world variations)
#  3. Effectively increase training dataset size
#
# IMPORTANT: Validation and test data are NOT augmented because they
#            must represent unseen data as realistically as possible.
# ─────────────────────────────────────────────────────────────────────────────

print(" TRAINING DATA AUGMENTATION:")
train_gen = ImageDataGenerator(
    # MobileNetV2-specific preprocessing
    # Converts pixel values to the format used during ImageNet pretraining
    # (values scaled to [-1, 1] instead of [0, 255])
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    
    # ═══════ GEOMETRIC AUGMENTATIONS ═══════
    
    # Random rotation: ±20 degrees
    # WHY?: Dogs can be photographed from different angles; skin disease
    #       looks similar regardless of rotation. Teaches rotation invariance.
    rotation_range=20,
    
    # Random horizontal shift: ±15% of width
    # WHY?: Dogs can be positioned left/right in the frame. Displacement
    #       shouldn't change disease classification.
    width_shift_range=0.15,
    
    # Random vertical shift: ±15% of height
    # WHY?: Dogs can be positioned high/low in the frame.
    height_shift_range=0.15,
    
    # Random zoom: 0.2 means zoom out to 80% or in to 120% of original
    # WHY?: Different photograph distances show same disease differently.
    #       Model should handle various "zoom levels".
    zoom_range=0.2,
    
    # Horizontal flip: Mirror images left-to-right
    # WHY?: Skin disease on left side looks same as on right side.
    #       50% chance to flip teaches symmetry invariance.
    horizontal_flip=True,
    
    # ═══════ PHOTOMETRIC AUGMENTATIONS ═══════
    
    # Random brightness: 80% to 120% of original
    # WHY?: Photos taken in different lighting (sunny, cloudy, indoor).
    #       Model must work in various lighting conditions.
    brightness_range=[0.8, 1.2],
)

print("  ✓ Rotation: ±20°")
print("  ✓ Horizontal shift: ±15%")
print("  ✓ Vertical shift: ±15%")
print("  ✓ Zoom: ±20%")
print("  ✓ Horizontal flip: yes")
print("  ✓ Brightness: 80-120%")

# Validation data: NO AUGMENTATION
# Validation is used to measure generalization performance on unseen data.
# Augmentation would distort the measurement.
val_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
)
print("\n✓ Validation data: MobileNetV2 preprocessing only (no augmentation)")

# ─────────────────────────────────────────────────────────────────────────────
# LOAD TRAINING DATA
# ─────────────────────────────────────────────────────────────────────────────
print("\n📂 LOADING DATA:")

train_ds = train_gen.flow_from_directory(
    os.path.join(DATASET_PATH, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',  # One-hot encoded labels (e.g., [1,0,0,0])
    shuffle=True,              # Randomize order to improve learning
)
print(f"  ✓ Training: {train_ds.samples} images in {train_ds.classes.max() + 1} classes")

# Load validation data
val_ds = val_gen.flow_from_directory(
    os.path.join(DATASET_PATH, 'valid'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,  # Keep fixed order for consistency
)
print(f"  ✓ Validation: {val_ds.samples} images")

# ─────────────────────────────────────────────────────────────────────────────
# EXTRACT CLASS INFORMATION
# ─────────────────────────────────────────────────────────────────────────────
# This mapping is CRITICAL for integration with the Flask backend.
# When model predicts class 0, we need to know it means "Healthy Skin", etc.

NUM_CLASSES = len(train_ds.class_indices)
print(f"\n🏷️  CLASSES DETECTED: {NUM_CLASSES} disease categories\n")
print("CLASS INDEX MAPPING:")
print("─" * 50)

for class_name in sorted(train_ds.class_indices.keys()):
    class_idx = train_ds.class_indices[class_name]
    print(f"  [{class_idx}] {class_name}")

print("─" * 50)
print("\n💡 KEEP THIS MAPPING! It's needed for app.py prediction decoding.\n")

## 4. Build MobileNetV2 Transfer Learning Model

This section creates a transfer learning model. MobileNetV2 is used as a pretrained feature extractor, and new classification layers are added to adapt it to the dog skin disease dataset.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 4: BUILD TRANSFER LEARNING MODEL
# ═════════════════════════════════════════════════════════════════════════════
#
# TRANSFER LEARNING CONCEPT:
# • MobileNetV2 learned general image features from 1.3M ImageNet images
# • Instead of training from scratch, we reuse these features (layers 1-154)
# • Only the classification head (final layers) are trained for dog skin diseases
# • This dramatically reduces training time and requires fewer images
#
# ARCHITECTURE:
#  Input (224×224×3)
#    ↓
#  MobileNetV2 Base (pretrained, frozen initially)
#    ↓
#  Global Average Pooling (reduce spatial dimensions)
#    ↓
#  Dropout (0.3) + Dense(256) + Dropout(0.2)
#    ↓
#  Output Dense(NUM_CLASSES, softmax) → probability distribution
#
# ═════════════════════════════════════════════════════════════════════════════

print("🏗️  BUILDING MODEL ARCHITECTURE")
print("=" * 70)

# Load MobileNetV2 with pretrained ImageNet weights
# This model has already learned to detect edges, textures, shapes from
# millions of images, so we get a "head start" instead of training from scratch
base = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),     # 224×224 RGB images
    include_top=False,              # Remove original ImageNet classification layers
    weights='imagenet',             # Use pretrained ImageNet weights
)
print(f"✓ MobileNetV2 loaded with ImageNet weights")
print(f"  Total layers: {len(base.layers)}")
print(f"  Trainable parameters: {base.count_params():,}")

# Freeze the base model for Phase 1 training
# WHY FREEZE?: The base model already knows how to extract features.
#             We should train only the new classification head first.
#             This prevents "catastrophic forgetting" of pretrained knowledge.
base.trainable = False
print(f"\n🔒 Phase 1: Base model FROZEN (not trainable)")

# Define model input
inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))

# Pass inputs through frozen MobileNetV2
# training=False ensures batch normalization layers use learned statistics
# (not recalculated on each batch, which would distort pretrained values)
x = base(inputs, training=False)
print("✓ Input → MobileNetV2 Base (frozen)")

# Convert feature maps to 1D vector
# MobileNetV2 outputs shape (batch, 7, 7, 1280)
# GlobalAveragePooling2D reduces this to (batch, 1280)
# This is better than flattening because it:
#  • Reduces parameters (no massive fully-connected layer)
#  • Is more robust to slight spatial shifts
x = layers.GlobalAveragePooling2D()(x)
print("✓ GlobalAveragePooling2D → 1280 features")

# Regularization: Dropout layer 1
# Randomly deactivates 30% of neurons during training
# WHY?: Prevents co-adaptation (neurons relying too much on each other)
#       Improves generalization to new images
x = layers.Dropout(0.3)(x)
print("✓ Dropout(0.3)")

# Dense layer with ReLU activation
# Learns disease-specific feature combinations
# 256 units is a reasonable size for this task
# ReLU: max(0, x) - introduces non-linearity
x = layers.Dense(256, activation='relu')(x)
print("✓ Dense(256, relu)")

# Regularization: Dropout layer 2
# Additional dropout to further prevent overfitting
x = layers.Dropout(0.2)(x)
print("✓ Dropout(0.2)")

# Output layer
# Softmax produces probability distribution across NUM_CLASSES
# Example with 7 classes: [0.05, 0.1, 0.15, 0.2, 0.25, 0.15, 0.1]
# The class with highest probability is the prediction
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
print(f"✓ Dense({NUM_CLASSES}, softmax) → prediction probabilities")

# Combine all layers into a model
model = Model(inputs, outputs)
print(f"\n📊 Model created!")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# Compile: configure for training
model.compile(
    # Adam optimizer: adaptive learning rate, works well for most tasks
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    
    # categorical_crossentropy: standard loss for multi-class classification
    # Measures difference between predicted probabilities and true labels
    loss='categorical_crossentropy',
    
    # Accuracy: % of correctly classified images
    metrics=['accuracy'],
)

print("\n✓ Model compiled (Adam optimizer, learning_rate=1e-3)")
print("=" * 70 + "\n")

## 5. Phase 1 Training — Train Classification Head

In the first phase, MobileNetV2 remains frozen and only the newly added layers are trained. This lets the model learn the new disease classification task without immediately changing the pretrained feature extractor.


In [ ]:

# STEP 5: PHASE 1 TRAINING — TRAIN CLASSIFICATION HEAD ONLY

#
# STRATEGY:
# • MobileNetV2 base is FROZEN (not updated)
# • Only the new classification head learns
# • This is safer: prevents damaging pretrained weights
# • Allows the classification head to adapt to dog skin diseases
#
# TRAINING DETAILS:
# • Each epoch: model sees all training images once
# • Each batch: 32 images → weights updated
# • Validation: measure performance on held-out validation set
# • Early stopping: stop if validation accuracy plateaus
#

print("PHASE 1: TRAINING CLASSIFICATION HEAD (Base Frozen)")
print("=" * 70)

# Train the model on training data with validation monitoring
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,  # Maximum 10 epochs (but may stop early)
    callbacks=[
        # ─────────────────────────────────────────────────────────────
        # CALLBACK 1: Early Stopping
        # ─────────────────────────────────────────────────────────────
        # PURPOSE: Prevent overfitting by stopping training when validation
        #          accuracy stops improving
        # 
        # LOGIC:
        #  • Monitor validation accuracy (val_accuracy)
        #  • If no improvement for 3 consecutive epochs (patience=3), stop
        #  • Restore the best weights from training
        #
        # EXAMPLE:
        #  Epoch 1: val_acc = 0.70 ✓ new best
        #  Epoch 2: val_acc = 0.72 ✓ new best
        #  Epoch 3: val_acc = 0.71 (no improvement, count=1)
        #  Epoch 4: val_acc = 0.71 (no improvement, count=2)
        #  Epoch 5: val_acc = 0.70 (no improvement, count=3) → STOP!
        # ─────────────────────────────────────────────────────────────
        EarlyStopping(
            monitor='val_accuracy',
            patience=3,
            restore_best_weights=True,
            verbose=1
        ),
        
        # ─────────────────────────────────────────────────────────────
        # CALLBACK 2: Reduce Learning Rate on Plateau
        # ─────────────────────────────────────────────────────────────
        # PURPOSE: When training stalls, reduce learning rate to make
        #          finer, more careful weight updates
        #
        # LOGIC:
        #  • Monitor validation loss
        #  • If no improvement for 2 epochs (patience=2):
        #    - New learning rate = old * factor (0.5)
        #    - So if LR was 0.001, new LR = 0.0005
        #
        # WHY?: Higher learning rates make big jumps, might miss minima.
        #       Lower learning rates make small steps, more precise.
        # ─────────────────────────────────────────────────────────────
        ReduceLROnPlateau(
            monitor='val_loss',
            patience=2,
            factor=0.5,
            verbose=1
        ),
        
        # ─────────────────────────────────────────────────────────────
        # CALLBACK 3: Model Checkpoint
        # ─────────────────────────────────────────────────────────────
        # PURPOSE: Save the best model during training for later use
        #
        # Saves to: phase1_best.h5
        # Only overwrites file if validation accuracy improves
        # ─────────────────────────────────────────────────────────────
        ModelCheckpoint(
            'phase1_best.h5',
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        ),
    ]
)

# ─────────────────────────────────────────────────────────────────────────────
# TRAINING COMPLETE: Analyze Phase 1 Results
# ─────────────────────────────────────────────────────────────────────────────

best_val_acc_phase1 = max(history1.history['val_accuracy'])
final_train_acc = history1.history['accuracy'][-1]
final_val_acc = history1.history['val_accuracy'][-1]

print("\n" + "=" * 70)
print("PHASE 1 RESULTS:")
print(f"  Best validation accuracy  : {best_val_acc_phase1:.4f} ({best_val_acc_phase1:.2%})")
print(f"  Final training accuracy   : {final_train_acc:.4f} ({final_train_acc:.2%})")
print(f"  Final validation accuracy : {final_val_acc:.4f} ({final_val_acc:.2%})")

# Check for overfitting signs
# OVERFITTING WARNING: training_acc >> validation_acc means memorization
gap = final_train_acc - final_val_acc
if gap > 0.15:
    print(f"   Gap (train - val)      : {gap:.4f} (⚠️  possible overfitting)")
else:
    print(f"  ✓ Gap (train - val)       : {gap:.4f} (healthy)")

print("=" * 70 + "\n")

## 6. Phase 2 Fine-Tuning — Unfreeze Top Layers

In the second phase, the top 30 layers of MobileNetV2 are unfrozen. A smaller learning rate is used so the pretrained features can adapt to the dog skin disease dataset without being changed too aggressively.


In [ ]:

# STEP 6: PHASE 2 FINE-TUNING — UNFREEZE TOP LAYERS

#
# MOTIVATION FOR PHASE 2:
# After Phase 1, the classification head is well-trained. Now we can "unlock"
# the top layers of MobileNetV2 to adapt pretrained features to this specific
# disease classification task. This is called "fine-tuning".
#
# LAYER ARCHITECTURE (MobileNetV2 has ~154 layers):
#  Frozen in Phase 1:
#  ├─ Layers 1-30: Low-level features (edges, textures, simple shapes)
#  │                These are GENERAL and apply to any image task
#  │                Keep them frozen to preserve ImageNet knowledge
#  │
#  Unfrozen in Phase 2:
#  ├─ Layers 125-154: High-level features (dog anatomy, skin texture)
#  │                 These can be adapted to dog skin disease specifics
#  │                 These layers benefit from fine-tuning
#
# LEARNING RATES:
#  Phase 1: learning_rate = 1e-3 (0.001)  ← Aggressive (head starts from random)
#  Phase 2: learning_rate = 1e-5 (0.00001) ← Very conservative (preserve pretrained)
#           (100x smaller!) ← Fine adjustments only
#
# ═════════════════════════════════════════════════════════════════════════════

print(" PHASE 2: FINE-TUNING TOP LAYERS (Selective Unfreezing)")
print("=" * 70)

# Enable training for the MobileNetV2 base model
base.trainable = True
print(f"✓ Base model unlocked for training")

# Freeze the bottom 124 layers (keep them from Phase 1)
# Only the top 30 layers will be fine-tuned
freeze_count = 0
for i, layer in enumerate(base.layers):
    if i < len(base.layers) - 30:  # Bottom layers
        layer.trainable = False
        freeze_count += 1
    else:  # Top 30 layers
        layer.trainable = True

print(f"✓ Layers 1-{len(base.layers) - 30}: FROZEN (preserve ImageNet features)")
print(f"✓ Layers {len(base.layers) - 29}-{len(base.layers)}: UNFROZEN (fine-tuning)")

# Recompile model after changing trainable layers
# IMPORTANT: Must recompile whenever you change trainable status!
model.compile(
    # CRITICAL: Much smaller learning rate for fine-tuning
    # WHY 1e-5?: Pretrained weights are already good. We make tiny adjustments.
    #            If we used 1e-3 (Phase 1 rate), we'd destroy the fine features.
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f"\n✓ Model recompiled with smaller learning rate (1e-5)")
print(f"  This ensures gradual, careful adaptation of fine-tuned layers\n")

# Continue training for up to 10 more epochs
# Early stopping will probably stop this earlier
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    initial_epoch=len(history1.history['accuracy']),  # Continue from Phase 1
    callbacks=[
        # Same callbacks as Phase 1, with slightly adjusted patience
        EarlyStopping(
            monitor='val_accuracy',
            patience=4,  # More patience in Phase 2 (fine-tuning is gradual)
            restore_best_weights=True,
            verbose=1
        ),
        
        ReduceLROnPlateau(
            monitor='val_loss',
            patience=2,
            factor=0.5,
            verbose=1
        ),
        
        ModelCheckpoint(
            'final_best.h5',  # Save final model to different file
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        ),
    ]
)

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 COMPLETE: Analyze Results
# ─────────────────────────────────────────────────────────────────────────────

best_val_acc_phase2 = max(history2.history['val_accuracy'])
final_train_acc_p2 = history2.history['accuracy'][-1]
final_val_acc_p2 = history2.history['val_accuracy'][-1]

print("\n" + "=" * 70)
print("PHASE 2 RESULTS:")
print(f"  Best validation accuracy  : {best_val_acc_phase2:.4f} ({best_val_acc_phase2:.2%})")
print(f"  Final training accuracy   : {final_train_acc_p2:.4f} ({final_train_acc_p2:.2%})")
print(f"  Final validation accuracy : {final_val_acc_p2:.4f} ({final_val_acc_p2:.2%})")

# Compare Phase 1 vs Phase 2 improvement
improvement = best_val_acc_phase2 - best_val_acc_phase1
print(f"\n IMPROVEMENT FROM PHASE 1→2: +{improvement:.4f} ({improvement:.2%})")

if improvement > 0:
    print(f"  ✓ Fine-tuning helped! (+{improvement:.2%})")
else:
    print(f"  ℹ Fine-tuning didn't improve validation accuracy")
    print(f"     (This is okay - Phase 1 may have already converged)")

print("=" * 70 + "\n")

## 7. Visualise Training Performance and Save Model

This section plots training/validation accuracy and loss across both training phases. The final model is saved as an H5 file so it can be used later by the Flask prediction service.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 7: VISUALIZE TRAINING PERFORMANCE & SAVE MODEL
# ═════════════════════════════════════════════════════════════════════════════
#
# WHY VISUALIZE?:
# • Accuracy curve shows if model is learning
# • Loss curve reveals overfitting (train_loss vs val_loss divergence)
# • Clear separation line shows where fine-tuning started
#
# WHAT TO LOOK FOR:
# ✓ Good: Both curves decrease smoothly
# ✓ Good: Validation follows training (not random fluctuations)
# ✗ Bad: Validation curve increases (overfitting)
# ✗ Bad: Flat curves (not learning)
#
# ═════════════════════════════════════════════════════════════════════════════

print("📊 GENERATING TRAINING PERFORMANCE PLOTS")
print("=" * 70)

# Extract metrics from both training phases
phase1_acc = history1.history['val_accuracy']
phase2_acc = history2.history['val_accuracy']
phase1_tr_acc = history1.history['accuracy']
phase2_tr_acc = history2.history['accuracy']

# Combine both phases for continuous view
all_val_acc = phase1_acc + phase2_acc
all_tr_acc = phase1_tr_acc + phase2_tr_acc

# Extract loss values
phase1_loss = history1.history['val_loss']
phase2_loss = history2.history['val_loss']
phase1_tr_loss = history1.history['loss']
phase2_tr_loss = history2.history['loss']

# Combine losses
all_val_loss = phase1_loss + phase2_loss
all_tr_loss = phase1_tr_loss + phase2_tr_loss

# Create epoch numbers (e.g., [1, 2, 3, ..., 20])
epochs_total = range(1, len(all_val_acc) + 1)

# Mark the transition from Phase 1 to Phase 2
ft_start = len(phase1_acc)

print(f"✓ Total epochs: {len(all_val_acc)}")
print(f"  - Phase 1 (freeze): {ft_start} epochs")
print(f"  - Phase 2 (fine-tune): {len(phase2_acc)} epochs\n")

# ─────────────────────────────────────────────────────────────────────────────
# CREATE SIDE-BY-SIDE PLOTS: Accuracy and Loss
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PLOT 1: ACCURACY CURVE
# Shows how % of correct predictions changed over training
axes[0].plot(epochs_total, all_tr_acc, 
            marker='o', markersize=4, label='Training Accuracy', 
            color='steelblue', linewidth=2)
axes[0].plot(epochs_total, all_val_acc, 
            marker='s', markersize=4, label='Validation Accuracy', 
            color='green', linewidth=2)

# Vertical line to show where fine-tuning started
axes[0].axvline(x=ft_start, color='orange', linestyle='--', 
               linewidth=2, label='Fine-tuning starts')

axes[0].set_title('Accuracy Over Time', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Add annotations for key points
best_val_acc = max(all_val_acc)
best_epoch = all_val_acc.index(best_val_acc) + 1
axes[0].annotate(f'Best: {best_val_acc:.2%}', 
                xy=(best_epoch, best_val_acc),
                xytext=(best_epoch-2, best_val_acc-0.05),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='green'))

# PLOT 2: LOSS CURVE
# Shows how well predictions match true labels
# Lower loss = better predictions
axes[1].plot(epochs_total, all_tr_loss, 
            marker='o', markersize=4, label='Training Loss', 
            color='steelblue', linewidth=2)
axes[1].plot(epochs_total, all_val_loss, 
            marker='s', markersize=4, label='Validation Loss', 
            color='red', linewidth=2)

# Vertical line to show fine-tuning transition
axes[1].axvline(x=ft_start, color='orange', linestyle='--', 
               linewidth=2, label='Fine-tuning starts')

axes[1].set_title('Loss Over Time (Lower is Better)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Adjust layout and save
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
print("✓ Saved: training_curves.png")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# PERFORMANCE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print(" TRAINING SUMMARY:")
print(f"  Best validation accuracy (Phase 1): {max(phase1_acc):.4f}")
print(f"  Best validation accuracy (Phase 2): {max(phase2_acc):.4f}")
print(f"  Overall best validation accuracy  : {max(all_val_acc):.4f}")
print(f"  Achieved at epoch                 : {all_val_acc.index(max(all_val_acc)) + 1}")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# SAVE TRAINED MODEL
# ─────────────────────────────────────────────────────────────────────────────
# The model is saved in HDF5 format (.h5) which includes:
#  • Model architecture (layer definitions)
#  • Trained weights (all 500K+ parameters)
#  • Compilation settings
#
# This file is used by the Flask backend (app.py) for predictions

print("\n SAVING TRAINED MODEL:")

# Save the final model
model.save('dog_skin_model.h5')
print("✓ Saved: dog_skin_model.h5")

# Get file size for reference
import os
model_size_mb = os.path.getsize('dog_skin_model.h5') / (1024 * 1024)
print(f"  File size: {model_size_mb:.2f} MB")


print("\n" + "=" * 70)
print(" MODEL TRAINING COMPLETE!")
print("=" * 70)

## 8. Prepare Test Dataset

The test dataset is loaded separately and is not augmented. Shuffling is disabled so the true labels stay aligned with the predictions during evaluation.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 8: PREPARE TEST DATASET FOR FINAL EVALUATION
# ═════════════════════════════════════════════════════════════════════════════
#
# PURPOSE OF TEST SET:
# • Completely separate from training and validation data
# • Represents "real world" unseen images
# • Final measure of model generalization
# • Used for all official evaluation metrics
#
# IMPORTANT: Test set is NEVER used during training!
#            This ensures unbiased performance estimation.
#
# ═════════════════════════════════════════════════════════════════════════════

print(" PREPARING TEST DATASET FOR EVALUATION")
print("=" * 70)

# Build test data generator (NO augmentation, only preprocessing)
# Test images should be in their "natural" state to represent real deployment
test_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
)

# Load test images from the test folder
test_ds = test_gen.flow_from_directory(
    os.path.join(DATASET_PATH, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    
    #  CRITICAL: shuffle=False keeps image order consistent
    # If shuffle=True:
    #   - model.predict() returns predictions in shuffled order
    #   - test_ds.classes returns true labels in shuffled order
    #   - BUT they might not match (different shuffle orders!)
    # If shuffle=False:
    #   - predictions[i] matches classes[i] ✓
    shuffle=False,
)

# Create ordered list of class names for reporting
# Index matches both test_ds.classes and model predictions
CLASS_NAMES = [name for name, _ in sorted(test_ds.class_indices.items(), key=lambda x: x[1])]

print(f"\n✓ Test dataset loaded")
print(f"  Images: {test_ds.samples}")
print(f"  Classes: {len(CLASS_NAMES)}")
print(f"\n CLASS ORDER (for prediction interpretation):")
print("  " + "-" * 60)
for i, cls in enumerate(CLASS_NAMES):
    print(f"  [{i}] {cls}")
print("  " + "-" * 60)
print("\n This order is critical! When model predicts index 0,")
print(f"   it means: {CLASS_NAMES[0]}")
print("=" * 70 + "\n")

## 9. Test Set Prediction and Classification Metrics

This section evaluates the trained model on the test set using accuracy, precision, recall, and F1-score. These metrics provide a more detailed understanding than accuracy alone, especially if the dataset is imbalanced.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 9: RUN INFERENCE ON TEST SET & CALCULATE METRICS
# ═════════════════════════════════════════════════════════════════════════════
#
# METRICS EXPLAINED:
#
# 1. ACCURACY: (True Positives + True Negatives) / Total
#    Example: 950 correct out of 1000 = 95% accuracy
#    ✓ Good for balanced datasets
#    ✗ Misleading if dataset is imbalanced (e.g., 90% class A, 10% class B)
#
# 2. PRECISION (per-class): TP / (TP + FP)
#    "Of all images I said were 'Ringworm', how many actually were?"
#    Example: Predicted 100 as Ringworm, 85 correct = 85% precision
#    Important for: Avoiding false alarms (unnecessary vet visits)
#
# 3. RECALL (per-class): TP / (TP + FN)
#    "Of all images that ARE 'Ringworm', how many did I find?"
#    Example: 200 true Ringworm images, found 170 = 85% recall
#    Important for: Not missing diagnoses (disease goes untreated)
#
# 4. F1-SCORE: Harmonic mean of Precision and Recall
#    Balances both metrics (single number to compare models)
#
# 5. MACRO-AVERAGE: Average of per-class metrics (all classes weighted equally)
#    Example: If Precision is [0.90, 0.70, 0.85, 0.95]
#             Macro Precision = (0.90+0.70+0.85+0.95)/4 = 0.8525
#
# ═════════════════════════════════════════════════════════════════════════════

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)

print("🧪 TEST SET INFERENCE & EVALUATION")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: RUN MODEL PREDICTIONS ON ENTIRE TEST SET
# ─────────────────────────────────────────────────────────────────────────────

print("\n🔮 Running inference on test set...")
print(f"   Total images: {test_ds.samples}")

# verbose=1 shows progress bar
y_pred_probs = model.predict(test_ds, verbose=1)  # Shape: (n_images, NUM_CLASSES)
print(f"✓ Predictions shape: {y_pred_probs.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: CONVERT PROBABILITIES TO CLASS PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
# Model outputs softmax probabilities (e.g., [0.05, 0.1, 0.7, 0.15])
# We need to convert these to class indices (e.g., 2)

# argmax: returns the index of the maximum value
# Example: argmax([0.05, 0.1, 0.7, 0.15]) = 2 (index of 0.7)
y_pred = np.argmax(y_pred_probs, axis=1)
print(f"✓ Predicted classes: {y_pred.shape} (class indices)")

# Get true labels from test generator
y_true = test_ds.classes
print(f"✓ True classes: {y_true.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: CALCULATE OVERALL ACCURACY
# ─────────────────────────────────────────────────────────────────────────────
# Simple metric: % of images classified correctly

test_acc = accuracy_score(y_true, y_pred)
print(f"\n🎯 ACCURACY: {test_acc:.4f} ({test_acc:.2%})")
print(f"   Correctly classified: {(y_pred == y_true).sum()} / {len(y_true)}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: DETAILED CLASSIFICATION REPORT (Per-class metrics)
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("📊 DETAILED CLASSIFICATION REPORT (Per-Class)")
print("=" * 70)
print(classification_report(
    y_true, y_pred, 
    target_names=CLASS_NAMES, 
    digits=4,
    zero_division=0
))

# Explanation of output:
# precision: Of all predicted as this class, % that were correct
# recall:    Of all that ARE this class, % that we found
# f1-score:  Balance between precision and recall
# support:   Number of test images in this class

## 10. Confusion Matrix

The confusion matrix shows which classes the model predicts correctly and where it becomes confused. This is useful for identifying whether the model struggles with specific disease categories.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 10: CONFUSION MATRIX
# ═════════════════════════════════════════════════════════════════════════════
#
# INTERPRETATION:
# • Diagonal cells (top-left to bottom-right) = correct predictions ✓
# • Off-diagonal cells = misclassifications ✗
#
# EXAMPLE (4 classes):
#                          TRUE LABEL
#                    A     B     C     D
#              A [  95     2     1     2 ]
#              B [   3    92     3     2 ]
# PREDICTED    C [   1     2    88     9 ]  ← Confused with D (9 times)
#              D [   1     4     8    87 ]
#
# READING:
# • Row A: Of 100 A's, model predicted 95 correct, 2 as B, 1 as C, 2 as D
# • Column A: 95 truly A's predicted correctly, but 3+1+1=5 were predicted as other
#
# HIGH VALUES DIAGONAL = GOOD (correct predictions)
# HIGH VALUES OFF-DIAGONAL = BAD (model confused between these classes)
#
# ═════════════════════════════════════════════════════════════════════════════

print("\n  GENERATING CONFUSION MATRIX")
print("=" * 70)

# Create confusion matrix by comparing true vs predicted labels
cm = confusion_matrix(y_true, y_pred)
print(f"✓ Confusion matrix shape: {cm.shape}")
print(f"  Rows = predicted classes")
print(f"  Columns = true classes")

# Create visualization
fig, ax = plt.subplots(figsize=(max(10, NUM_CLASSES), max(8, NUM_CLASSES)))

# Plot confusion matrix with color scale
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(
    ax=ax, 
    cmap='Blues',           # Blue colors (0=white, high=dark blue)
    colorbar=True,          # Add color scale
    xticks_rotation=45      # Rotate x labels for readability
)

# Enhance the plot
ax.set_title('Confusion Matrix — Test Set\n(Diagonal = Correct, Off-diagonal = Errors)', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('True Label', fontsize=12, fontweight='bold')
ax.set_ylabel('Predicted Label', fontsize=12, fontweight='bold')

# Make the grid more visible
ax.grid(False)
ax.xaxis.set_ticks_position('bottom')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✓ Saved: confusion_matrix.png")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# ANALYZE CONFUSION PATTERNS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print(" CONFUSION ANALYSIS")
print("=" * 70)

# Identify classes with highest accuracy (diagonal values)
for i in range(NUM_CLASSES):
    correct = cm[i, i]
    total = cm[i, :].sum()
    if total > 0:
        class_accuracy = correct / total * 100
        print(f"{CLASS_NAMES[i]:<30s}: {class_accuracy:6.2f}% ({correct}/{total})")

# Find most common confusion (largest off-diagonal value)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)  # Set diagonal to 0
if cm_no_diag.max() > 0:
    max_confusion_idx = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
    pred_class = CLASS_NAMES[max_confusion_idx[0]]
    true_class = CLASS_NAMES[max_confusion_idx[1]]
    confusion_count = cm_no_diag[max_confusion_idx]
    
    print(f"\n Most common confusion:")
    print(f"   {true_class} → misclassified as {pred_class} ({int(confusion_count)} times)")

print("=" * 70 + "\n")

## 11. ROC Curve and AUC Evaluation

This section computes one-vs-rest ROC curves for each class. AUC-ROC helps evaluate how well the model separates each class from the others.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 11: ROC CURVE AND AUC EVALUATION
# ═════════════════════════════════════════════════════════════════════════════
#
# ROC = Receiver Operating Characteristic
# AUC = Area Under Curve
#
# WHAT IS ROC CURVE?:
# Shows the trade-off between True Positive Rate (sensitivity) and
# False Positive Rate (1 - specificity) as the classification threshold changes.
#
# WHY USE ROC/AUC?:
# • Doesn't depend on class balance
# • Shows performance across all thresholds
# • AUC = 0.5 means random guessing (diagonal line)
# • AUC = 1.0 means perfect classification
# • AUC = 0.7-0.8 is good, 0.8+ is excellent
#
# ONE-VS-REST:
# For multi-class (e.g., 7 diseases), calculate ROC for each class:
# • Class 0 (Ringworm) vs rest (all other diseases)
# • Class 1 (Dermatitis) vs rest
# • ... and so on
#
# EXAMPLE INTERPRETATION:
# • Model has AUC=0.92 for Ringworm detection
#   → Very good at separating Ringworm from other conditions
# • Model has AUC=0.75 for Uncertain cases
#   → Moderate difficulty distinguishing uncertain cases
#
# ═════════════════════════════════════════════════════════════════════════════

from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

print(" ROC CURVE & AUC ANALYSIS")
print("=" * 70)

# Convert multi-class labels to binary format for one-vs-rest ROC calculation
# Example with 4 classes:
#  Class 0 → [1, 0, 0, 0]
#  Class 1 → [0, 1, 0, 0]
#  Class 2 → [0, 0, 1, 0]
#  Class 3 → [0, 0, 0, 1]
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

print(f"Converted labels to binary format: {y_true_bin.shape}")

# Create ROC curves for each class
plt.figure(figsize=(10, 7))
auc_scores = {}

print(f"\n Computing ROC curves for each class...\n")

for i, cls_name in enumerate(CLASS_NAMES):
    # Calculate FPR (False Positive Rate) and TPR (True Positive Rate)
    # for different classification thresholds
    fpr, tpr, thresholds = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    
    # AUC = Area Under the ROC Curve
    # Calculated as the area between the curve and x-axis
    roc_auc = auc(fpr, tpr)
    auc_scores[cls_name] = roc_auc
    
    # Plot this class's ROC curve
    plt.plot(
        fpr, tpr, 
        label=f'{cls_name} (AUC = {roc_auc:.3f})',
        linewidth=2,
        marker='.' if i % 2 == 0 else None
    )
    
    # Print class-specific AUC
    auc_quality = "Excellent" if roc_auc > 0.8 else "Good" if roc_auc > 0.7 else "Fair"
    print(f"  [{i}] {cls_name:<30s} AUC = {roc_auc:.4f} ({auc_quality})")

# Plot reference line: random classifier
# A model with no discriminative ability (random guessing) produces AUC = 0.5
# This is represented by the diagonal line from (0,0) to (1,1)
plt.plot(
    [0, 1], [0, 1], 
    'k--', 
    linewidth=2,
    label='Random Classifier (AUC = 0.500)',
    alpha=0.7
)

# Formatting
plt.xlabel('False Positive Rate', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=11, fontweight='bold')
plt.title('ROC Curves — One-vs-Rest Classification (Test Set)', 
         fontsize=12, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)

# Set axis limits
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: roc_curves.png\n")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE MACRO-AVERAGE AUC
# ─────────────────────────────────────────────────────────────────────────────
# Treats each class equally (doesn't account for class imbalance)

macro_auc = sum(auc_scores.values()) / len(auc_scores)

print("=" * 70)
print(" AUC SUMMARY")
print("=" * 70)
print(f"Macro-average AUC: {macro_auc:.4f} ({macro_auc:.2%})")

# Interpret macro AUC
if macro_auc > 0.9:
    print("✓ EXCELLENT: Model is highly discriminative across all classes")
elif macro_auc > 0.8:
    print("✓ GOOD: Model shows good discriminative ability")
elif macro_auc > 0.7:
    print("ℹ  FAIR: Model has acceptable discriminative ability")
else:
    print(" POOR: Model needs improvement")

print("=" * 70 + "\n")

## 12. Final Evaluation Summary 
This final section summarises the test performance the generated evaluation images. These outputs can be used as evidence in the testing/evaluation section of the project report.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# STEP 12: FINAL EVALUATION SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
#
# METRICS EXPLAINED:
#
# ACCURACY: (Correct predictions) / (Total predictions)
#   • Single metric representing overall performance
#   • Useful for balanced datasets
#   • Can be misleading for imbalanced datasets
#
# MACRO PRECISION: Average precision across all classes
#   • Class 0 precision: 0.90
#   • Class 1 precision: 0.75
#   • Class 2 precision: 0.88
#   • Average: (0.90 + 0.75 + 0.88) / 3 = 0.84
#   • Treats all classes equally
#
# MACRO RECALL: Average recall across all classes
#   • How many instances of each class we detected (on average)
#
# MACRO F1-SCORE: Average F1 across all classes
#   • Harmonic mean of precision and recall
#   • Single balanced metric
#
# MACRO AUC-ROC: Average AUC across all classes
#   • ROC performance averaged across classes
#
# ═════════════════════════════════════════════════════════════════════════════

from sklearn.metrics import precision_score, recall_score, f1_score

print(" FINAL MODEL EVALUATION SUMMARY")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# CALCULATE MACRO-AVERAGE METRICS
# ─────────────────────────────────────────────────────────────────────────────
# "Macro" = average for each class equally (ignore class frequency)
# Useful for imbalanced datasets where some classes have fewer examples

macro_p = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_r = recall_score(y_true, y_pred, average='macro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n OVERALL METRICS:")
print("─" * 70)
print(f"Test Accuracy        : {test_acc:.4f}  ({test_acc:.2%})")
print(f"Macro Precision      : {macro_p:.4f}  ({macro_p:.2%})")
print(f"Macro Recall         : {macro_r:.4f}  ({macro_r:.2%})")
print(f"Macro F1-Score       : {macro_f1:.4f}  ({macro_f1:.2%})")
print(f"Macro AUC-ROC        : {macro_auc:.4f}  ({macro_auc:.2%})")
print("─" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# INTERPRET RESULTS
# ─────────────────────────────────────────────────────────────────────────────

print("\n INTERPRETATION:")

if test_acc > 0.85:
    print(f" Accuracy ({test_acc:.2%}) is EXCELLENT")
elif test_acc > 0.75:
    print(f" Accuracy ({test_acc:.2%}) is GOOD")
elif test_acc > 0.65:
    print(f"ℹ  Accuracy ({test_acc:.2%}) is ACCEPTABLE")
else:
    print(f"  Accuracy ({test_acc:.2%}) needs improvement")

if macro_p > macro_r:
    print(f"  Model is more conservative (higher precision than recall)")
    print(f"  → Less false positives, but might miss some cases")
else:
    print(f"  Model is more aggressive (higher recall than precision)")
    print(f"   Fewer missed cases, but more false alarms")

print("\n" + "=" * 70)
print(" EVALUATION COMPLETE!")
print("=" * 70)


plot_files = [
    'training_curves.png',
    'confusion_matrix.png',
    'roc_curves.png'
]

print("─" * 70)

print("\n MODEL TRAINING AND EVALUATION COMPLETE!")
print("=" * 70 + "\n")